<img src="https://devra.ai/analyst/notebook/4514/image.jpg" style="width: 100%; height: auto;" />

<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">
  <div style="font-size:150%; color:#FEE100"><b>Social Media Performance and Engagement Analysis</b></div>
  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div>
</div>

# Table of Contents

1. [Introduction and Data Loading](#Introduction-and-Data-Loading)
2. [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
3. [Exploratory Data Analysis](#Exploratory-Data-Analysis)
4. [Predictive Modeling](#Predictive-Modeling)
5. [Summary and Future Work](#Summary-and-Future-Work)

If you find this notebook useful, consider upvoting it.

In [ ]:
# Import required libraries and suppress warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use('Agg')  # if using full matplotlib
import matplotlib.pyplot as plt
plt.switch_backend('Agg')  # if only using plt from matplotlib

import warnings
warnings.filterwarnings('ignore')

# For inline plotting in Jupyter Notebook
%matplotlib inline

# For predictive modeling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.inspection import permutation_importance

# Set seaborn style
sns.set(style='whitegrid')

# Introduction and Data Loading

The social media performance dataset provides a rich view of different posts across various platforms. It includes information ranging from textual data (platform, content_type, topic, etc.) to numerical performance metrics (views, likes, comments, shares, engagement_rate). This is a great example of heterogeneous data that can benefit from both descriptive and predictive analytical approaches.

Let's load the dataset and take a quick look.

In [ ]:
# Load the data
data_path = '/kaggle/input/social-media-performance-and-engagement-data/social_media_performance.csv'
df = pd.read_csv(data_path, delimiter=',', encoding='ascii')

# Convert 'post_datetime' from string to datetime if possible
try:
    df['post_datetime'] = pd.to_datetime(df['post_datetime'])
except Exception as e:
    # Logging the error because other users might face similar issues
    print('Error converting post_datetime to datetime:', e)

print('Data loaded successfully. Shape:', df.shape)
df.head()

# Data Cleaning and Preprocessing

In this section, we clean the dataset and perform any necessary preprocessing steps. Issues like duplicate records, missing values, or incorrect data types are addressed. Here, we explicitly convert the date column and perform a basic check on the numerical columns.

In [ ]:
# Basic cleaning: remove duplicate rows and handle missing values if they exist
df.drop_duplicates(inplace=True)

# Check for missing values and display a summary
missing_summary = df.isnull().sum()
print('Missing values in each column:')
print(missing_summary)

# For demonstration, simply fill missing numeric values with median and others with mode
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)

# Convert date type if it wasn't already converted
if df['post_datetime'].dtype == 'object':
    try:
        df['post_datetime'] = pd.to_datetime(df['post_datetime'])
    except Exception as e:
        print('Error converting post_datetime:', e)

print('Data cleaning and preprocessing complete.')

# Exploratory Data Analysis

The following visualizations provide insights into the dataset. We employ various methods such as histograms, box plots, and heatmaps to gain a comprehensive understanding of the relationships and distributions present.

Below are several plots including the relationship among numeric features, distribution of sentiment scores, and breakdowns by categorical variables.

In [ ]:
# Visualizations

# Numeric dataframe for correlation analysis
numeric_df = df.select_dtypes(include=[np.number])

# Correlation heatmap if there are four or more numeric columns
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(10, 8))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Heatmap of Numeric Features')
    plt.tight_layout()
    plt.show()

# Pair plot of numeric features (if dataset size permits)
if numeric_df.shape[1] >= 2:
    sns.pairplot(numeric_df)
    plt.suptitle('Pair Plot of Numeric Features', y=1.02)
    plt.show()

# Histogram of sentiment scores
plt.figure(figsize=(8, 6))
sns.histplot(df['sentiment_score'], kde=True, bins=30)
plt.title('Distribution of Sentiment Scores')
plt.show()

# Box plot for engagement_rate by content_type
plt.figure(figsize=(10, 6))
sns.boxplot(x='content_type', y='engagement_rate', data=df)
plt.title('Engagement Rate by Content Type')
plt.xticks(rotation=45)
plt.show()

# Count plot (pie chart-like) for posts per platform
plt.figure(figsize=(8, 6))
sns.countplot(y='platform', data=df, order=df['platform'].value_counts().index)
plt.title('Number of Posts per Platform')
plt.show()

# Predictive Modeling

We now try to predict the binary variable 'is_viral' based on numerical and selected categorical features. In this example, a logistic regression model is built. The features used include views, likes, comments, shares, engagement_rate and sentiment_score. Note that in practice, one might consider a more complex model and a more rigorous feature selection/engineering process.

Let's build the model and evaluate its performance using accuracy score and a confusion matrix.

In [ ]:
# Prepare data for modeling

# For a quick demonstration, we use a subset of features
features = ['views', 'likes', 'comments', 'shares', 'engagement_rate', 'sentiment_score']
target = 'is_viral'

# Ensure there are no missing values in these columns
model_df = df[features + [target]].dropna()

# Split data into train and test sets
X = model_df[features]
y = model_df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions on test set
y_pred = model.predict(X_test)

# Calculate prediction accuracy
accuracy = accuracy_score(y_test, y_pred)
print('Logistic Regression Model Accuracy:', accuracy)

# Plotting the confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# ROC Curve
y_prob = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='ROC Curve (area = {:.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

# Permutation importance
perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
sorted_idx = perm_importance.importances_mean.argsort()
plt.figure(figsize=(8, 6))
plt.barh(np.array(features)[sorted_idx], perm_importance.importances_mean[sorted_idx])
plt.xlabel('Permutation Importance')
plt.title('Feature Importance based on Permutation')
plt.show()

# Summary and Future Work

This notebook presented a comprehensive exploration of the social media performance dataset. We started by cleaning the data and ensuring the proper conversion of date types. A wide range of visualizations provided insights into distribution patterns and feature relationships. Finally, a logistic regression model was used to predict the viral nature of a post, and its performance was evaluated.

Merits of this approach include:
- A thorough cleaning and automated handling of missing values
- A variety of visualization techniques to understand multi-dimensional data
- A baseline predictive model that demonstrates potential for further model improvement

For future analysis, one may consider:
- Conducting feature engineering on text-based features such as hashtags and topics
- Experimenting with more sophisticated models (e.g., tree-based ensemble methods or neural networks)
- Performing cross-validation and hyperparameter tuning for optimized performance

If you find this notebook useful, please consider upvoting it.